# 01 · Scenes and wind

Handoff §3 Steps 1 and 4, for **one site** chosen by the environment variable `DUNESIGHT_SITE` (default `seih_al_hama`; boxes live in `pipeline/sites.json`).

What this notebook does, in order:
1. **Access tests**: Earth Engine returns a Landsat scene over the box; earthaccess finds an HLS granule over the box.
2. **Landsat candidate table** (WRS-2 path/row from `sites.json`): Landsat 7 ETM+ before the SLC failure (31 May 2003), Landsat 8/9 OLI 2013-2025. Per scene: ID, date, sun angles, scene cloud, box coverage, in-box cloud / dilated cloud / cirrus / shadow from `QA_PIXEL`, a check that band 8 (15 m pan) is present and valid, a texture index and a haze indicator.
3. **One scene per year** for May and for November under the rule "reject if more than 1 % cloud/shadow inside the box", every rejection logged, B8 thumbnails of both sets, and a recommended month. Written to `pipeline/outputs/<site>/` as a **proposal awaiting confirmation**. Nothing full-resolution is downloaded here (that is notebook 02).
4. **Wind**: Fryberger drift potential from MERRA-2 hourly 10 m wind, 2000-2024, with unit tests of the maths, then a NASA POWER cross-check.
5. Writes `public/data/<site>/wind_summary.json` (real values) and refreshes `public/data/sites.json`.

Run headless: `npm run pipeline` (all verified sites) or `DUNESIGHT_SITE=seih_al_hama .venv/bin/jupyter nbconvert --to notebook --execute 01_scenes_and_wind.ipynb --output-dir runs/seih_al_hama` from `pipeline/`. The executed copy with outputs is kept in `pipeline/runs/<site>/`; this source notebook stays without outputs.

Earth Engine results are cached in `pipeline/raw/<site>/` (gitignored). Set `DUNESIGHT_REFRESH=1` to re-query everything.

In [ ]:
# WHAT THIS DOES: finds the pipeline folder, imports the shared dunesight helpers, picks the site from
# DUNESIGHT_SITE, sets the output folders, and loads .env (only reports whether each secret is present).
import json, os, sys, time, math, io, datetime as dt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

PIPELINE = next(p for p in [Path.cwd(), Path.cwd() / "pipeline", Path.cwd().parent, Path.cwd().parent.parent]
                if (p / "dunesight").is_dir())
sys.path.insert(0, str(PIPELINE))

import numpy as np
import pandas as pd
import requests
import matplotlib
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from dunesight import config, site_index, wind, landsat, merra2

SITE = config.get_site()                      # raises if the site is unknown or not verified
SID = SITE["id"]
BBOX = config.bbox_list(SITE)                 # [W, S, E, N]
LAT_C, LON_C = config.centroid(SITE)
OUT = config.outputs_dir(SID); OUT.mkdir(parents=True, exist_ok=True)
RAW = config.raw_dir(SID); RAW.mkdir(parents=True, exist_ok=True)
PUB = config.public_dir(SID); PUB.mkdir(parents=True, exist_ok=True)
REFRESH = os.environ.get("DUNESIGHT_REFRESH", "") == "1"
load_dotenv(config.ROOT / ".env")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

print(f"site: {SID}  ({SITE['name']})  verified={SITE['verified']}")
print(f"bbox W,S,E,N: {BBOX}   centroid: {LAT_C:.4f} N, {LON_C:.4f} E   EPSG:{SITE['epsg']}   WRS-2 {SITE['wrs_path']}/{SITE['wrs_row']:03d}")
print("box source:", SITE["source"])
print("credentials present:", {k: bool(os.getenv(k)) for k in ("EE_PROJECT", "EARTHDATA_USERNAME", "EARTHDATA_PASSWORD")})
print("cache refresh forced:", REFRESH)

## 1 · Access tests (handoff §5.2)

In [ ]:
# WHAT THIS DOES: initialises Earth Engine (project id from .env, not printed) and prints one Landsat 8
# Collection 2 Tier 1 scene over the box, as the §5.2 access test.
ee = landsat.ee_init(config.ROOT)
BOX = ee.Geometry.Rectangle(BBOX, "EPSG:4326", False)
first = (ee.ImageCollection("LANDSAT/LC08/C02/T1").filterBounds(BOX)
         .filter(ee.Filter.eq("WRS_PATH", SITE["wrs_path"])).filter(ee.Filter.eq("WRS_ROW", SITE["wrs_row"]))
         .filterDate("2024-05-01", "2024-06-01").first())
info = first.toDictionary(["LANDSAT_PRODUCT_ID", "DATE_ACQUIRED"]).getInfo()
print("Earth Engine OK. One Landsat scene over the box:", info["LANDSAT_PRODUCT_ID"], info["DATE_ACQUIRED"])

In [ ]:
# WHAT THIS DOES: logs in to NASA Earthdata with earthaccess (credentials from .env, never printed) and finds
# one HLS L30 granule over the box, as the §5.2 access test (search only; nothing is downloaded).
import logging, earthaccess
logging.getLogger("earthaccess").setLevel(logging.WARNING)
_auth = earthaccess.login(strategy="environment")
print("Earthdata login authenticated:", bool(_auth and _auth.authenticated))
gran = earthaccess.search_data(short_name="HLSL30", bounding_box=tuple(BBOX), temporal=("2024-05-01", "2024-05-31"), count=5)
print(f"HLSL30 granules over the box in May 2024 (first {len(gran)} shown):")
for g in gran:
    print("  ", g["meta"]["native-id"])
del _auth

## 2 · Landsat candidate scenes

**Collections** (Earth Engine, Collection 2 Level-1 Tier 1): `LANDSAT/LE07/C02/T1` from 2000-01-01 up to the scan-line-corrector failure on **2003-05-31** (later L7 scenes have SLC-off gaps and are listed but excluded), and `LANDSAT/LC08/C02/T1` + `LANDSAT/LC09/C02/T1` for 2013-2025. Only WRS-2 path/row from `sites.json` (160/043 here) so every scene shares one viewing geometry.

**In-box cloud** uses the `QA_PIXEL` bits over the valid (non-fill) 30 m pixels inside the box:
bit 0 fill, **bit 1 dilated cloud, bit 2 cirrus (OLI only; always 0 on ETM+), bit 3 cloud, bit 4 cloud shadow**. `pct_any_cloud_shadow` is the share of pixels with any of bits 1-4 set; it is the number the 1 % rule uses. Dilated cloud is a buffer around detected clouds, so including it is deliberately conservative. These are the USGS automatic (CFMask) flags: they miss thin dust haze, so a haze indicator is added below.

**Coverage**: the scene footprint contains the box *and* every 30 m QA pixel and 15 m B8 pixel in the box is valid (non-fill / non-zero).

**Band 8 check**: share of non-zero B8 pixels at 15 m, DN min/max/percentiles, and the share flagged saturated in `QA_RADSAT` bit 7 (band 8).

**Texture index** (for feature tracking): standard deviation of the B8 top-of-atmosphere reflectance after removing a 150 m-radius moving mean, divided by the mean reflectance. It measures 15 m-scale contrast (dune slip faces, shading). Higher is better for tracking. It is a relative index: ETM+ is 8-bit and OLI 12-bit (delivered as 16-bit), so compare it within a sensor.

**Haze indicator**: MERRA-2 hourly aerosol optical depth at 550 nm (total `TOTEXTTAU` and dust `DUEXTTAU`, `NASA/GSFC/MERRA/aer/2`) in the grid cell containing the box, for the hour of acquisition. It is a ~50 km model value, so an indicator of regional dust, not a measurement over the box. (The cell is chosen with the grid correction explained in section 4.)

In [ ]:
# WHAT THIS DOES: computes, on Earth Engine, the per-scene statistics described above for every path/row scene in
# the windows (one request per sensor-year, run in parallel), and caches the table in pipeline/raw/<site>/.
SCENE_CACHE = RAW / "landsat_scene_stats.csv"
EPSG = SITE["epsg"]
MERRA_TRANSFORM = ee.ImageCollection("NASA/GSFC/MERRA/slv/2").first().select("U10M").projection().getInfo()["transform"]
_cell = merra2.native_cell(LAT_C, LON_C)
AER_PT = ee.Geometry.Point(merra2.ee_pixel_centre(MERRA_TRANSFORM, _cell["i"], _cell["j"]))

def pr(cid):
    return (ee.ImageCollection(cid).filter(ee.Filter.eq("WRS_PATH", SITE["wrs_path"]))
            .filter(ee.Filter.eq("WRS_ROW", SITE["wrs_row"])))

WINDOWS = [("LANDSAT/LE07/C02/T1", y, f"{y}-01-01", f"{y + 1}-01-01" if y < 2003 else landsat.L7_SLC_FAILURE) for y in range(2000, 2004)]
WINDOWS += [(cid, y, f"{y}-01-01", f"{y + 1}-01-01") for cid in ("LANDSAT/LC08/C02/T1", "LANDSAT/LC09/C02/T1") for y in range(2013, 2026)]

def stats_for(w):
    cid, y, start, end = w
    fc = pr(cid).filterDate(start, end).map(lambda im: landsat.scene_stats(ee, im, BOX, EPSG, AER_PT, MERRA_TRANSFORM))
    for attempt in range(4):
        try:
            rows = [f["properties"] for f in fc.getInfo()["features"]]
            for r in rows:
                r["collection"] = cid
            return rows
        except Exception as e:  # transient EE errors: retry with back-off
            if attempt == 3:
                raise
            time.sleep(5 * (attempt + 1))

if SCENE_CACHE.exists() and not REFRESH:
    scenes = pd.read_csv(SCENE_CACHE)
    print(f"loaded cached scene statistics: {len(scenes)} scenes ({SCENE_CACHE.relative_to(config.ROOT)})")
else:
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        scenes = pd.DataFrame([r for rows in ex.map(stats_for, WINDOWS) for r in rows])
    scenes.to_csv(SCENE_CACHE, index=False)
    print(f"queried Earth Engine: {len(scenes)} scenes in {time.time() - t0:.0f} s")

scenes = landsat.derive(scenes)
scenes["date"] = pd.to_datetime(scenes["date"])
scenes["year"], scenes["month"] = scenes["date"].dt.year, scenes["date"].dt.month
scenes["sensor"] = scenes["collection"].map(landsat.COLLECTIONS)
scenes = scenes.sort_values("date").reset_index(drop=True)
print(scenes.groupby("sensor").agg(n=("scene_id", "size"), first=("date", "min"), last=("date", "max")))

In [ ]:
# WHAT THIS DOES: lists the Landsat 7 scenes on this path/row from the SLC failure (2003-05-31) onwards. They are
# excluded (scan-line gaps) and logged here and in the candidates CSV, without computing statistics for them.
slc = pr("LANDSAT/LE07/C02/T1").filterDate(landsat.L7_SLC_FAILURE, "2026-01-01")
slc_off = pd.DataFrame({"scene_id": slc.aggregate_array("LANDSAT_PRODUCT_ID").getInfo(),
                        "date": pd.to_datetime(slc.aggregate_array("DATE_ACQUIRED").getInfo())})
slc_off["collection"], slc_off["sensor"] = "LANDSAT/LE07/C02/T1", "Landsat 7 ETM+"
slc_off["excluded_reason"] = "Landsat 7 SLC-off (acquired on/after 2003-05-31)"
print(f"Excluded Landsat 7 SLC-off scenes on {SITE['wrs_path']}/{SITE['wrs_row']:03d}: {len(slc_off)} "
      f"({slc_off.date.min().date()} to {slc_off.date.max().date()})")
print("Last pre-SLC-failure L7 scene kept:", scenes[scenes.sensor == "Landsat 7 ETM+"].iloc[-1][["scene_id", "date"]].tolist())

In [ ]:
# WHAT THIS DOES: derives the pass/fail flags (box coverage, B8 validity, the 1 % cloud/shadow rule, the haze test) for every
# candidate, prints the B8 validity check, and summarises in-box cloud by sensor.
CLOUD_MAX_PCT = 1.0
scenes["covers_box"] = (scenes.footprint_contains_box == True) & (scenes.qa_valid_frac >= 0.999) & (scenes.b8_valid_frac >= 0.999)
scenes["b8_ok"] = (scenes.b8_valid_frac >= 0.999) & (scenes.b8_saturated_pct < 0.1) & (scenes.b8_p98 > scenes.b8_p2)
scenes["cloud_ok"] = scenes.pct_any_cloud_shadow <= CLOUD_MAX_PCT
# haze/dust test (QA_PIXEL does not flag dust): texture below half the median of the same calendar month's clear,
# covering scenes from the same sensor family
HAZE_FRACTION = 0.5
scenes["family"] = np.where(scenes.sensor == "Landsat 7 ETM+", "ETM+", "OLI")
_clear = scenes[scenes.covers_box & scenes.b8_ok & scenes.cloud_ok]
TEX_REF = _clear.groupby(["month", "family"]).b8_texture.median()
scenes["haze_ok"] = [t >= HAZE_FRACTION * TEX_REF.get((m, f), np.nan) for t, m, f in zip(scenes.b8_texture, scenes.month, scenes.family)]
print("median texture of clear covering scenes (month, family):")
print(TEX_REF.unstack().loc[[5, 11]].round(4))

print("Band 8 presence/validity over the box (all candidates):")
print(scenes.groupby("sensor").agg(n=("scene_id", "size"),
      b8_valid_frac_min=("b8_valid_frac", "min"), b8_sat_pct_max=("b8_saturated_pct", "max"),
      b8_dn_min=("b8_min", "min"), b8_dn_max=("b8_max", "max"), b8_ok=("b8_ok", "sum"), covers_box=("covers_box", "sum")))
bad = scenes[~scenes.b8_ok | ~scenes.covers_box]
print(f"\nScenes failing coverage or B8 checks: {len(bad)}")
if len(bad):
    print(bad[["scene_id", "date", "footprint_contains_box", "qa_valid_frac", "b8_valid_frac", "b8_saturated_pct"]].to_string(index=False))
print(f"\nScenes with in-box cloud/shadow <= {CLOUD_MAX_PCT} %: {int(scenes.cloud_ok.sum())} of {len(scenes)}")
print("In-box cloud/shadow % by sensor (any of bits 1-4):")
print(scenes.groupby("sensor").pct_any_cloud_shadow.describe(percentiles=[.5, .75, .9]).round(2))

In [ ]:
# WHAT THIS DOES: cross-checks the Earth Engine scene list against the Phase 1 USGS STAC list
# (docs/research/landsat_candidates.csv, read only) and compares Earth Engine in-box cloud with STAC scene cloud.
stac_path = config.ROOT / "docs" / "research" / "landsat_candidates.csv"
if stac_path.exists():
    stac = pd.read_csv(stac_path)
    stac = stac[(stac.wrs_path == SITE["wrs_path"]) & (stac.wrs_row == SITE["wrs_row"]) & (stac.collection_category == "T1")]
    ee_ids, stac_ids = set(scenes.scene_id), set(stac.scene_id)
    print(f"STAC T1 {SITE['wrs_path']}/{SITE['wrs_row']:03d} scenes: {len(stac_ids)}; Earth Engine: {len(ee_ids)}; in both: {len(ee_ids & stac_ids)}")
    print("only in Earth Engine:", sorted(ee_ids - stac_ids)[:20])
    print("only in STAC        :", sorted(stac_ids - ee_ids)[:20])
    both = scenes.merge(stac[["scene_id", "cloud_cover_scene"]].rename(columns={"cloud_cover_scene": "stac_cloud"}), on="scene_id")
    print("max |EE CLOUD_COVER - STAC eo:cloud_cover|:", round((both.cloud_cover_scene - both.stac_cloud).abs().max(), 3))
    lowscene_cloudy_box = scenes[(scenes.cloud_cover_scene <= 1) & ~scenes.cloud_ok]
    print(f"scenes with scene cloud <= 1 % but in-box cloud/shadow > 1 %: {len(lowscene_cloudy_box)}")
    print(lowscene_cloudy_box[["scene_id", "date", "cloud_cover_scene", "pct_any_cloud_shadow"]].assign(date=lowscene_cloudy_box.date.dt.date).round(2).to_string(index=False))
else:
    print("docs/research/landsat_candidates.csv not present; cross-check skipped")

In [ ]:
# WHAT THIS DOES: writes the full candidate table (every scene, plus the excluded SLC-off L7 scenes) to
# pipeline/outputs/<site>/landsat_candidates.csv.
cols = ["scene_id", "ee_id", "collection", "sensor", "date", "year", "month", "sun_elevation", "sun_azimuth",
        "cloud_cover_scene", "cloud_cover_land", "footprint_contains_box", "qa_valid_frac", "covers_box",
        "pct_cloud", "pct_dilated_cloud", "pct_cirrus", "pct_cloud_shadow", "pct_any_cloud_shadow", "cloud_ok", "haze_ok",
        "b8_valid_frac", "b8_min", "b8_max", "b8_p2", "b8_p98", "b8_saturated_pct", "b8_ok",
        "b8_texture", "family", "b8_toa_mean", "merra2_aod550_total", "merra2_aod550_dust"]
cand = pd.concat([scenes[cols].assign(excluded_reason=""), slc_off], ignore_index=True).sort_values("date")
cand["excluded_reason"] = cand["excluded_reason"].fillna("")
cand["date"] = cand["date"].dt.strftime("%Y-%m-%d")
cand.round(4).to_csv(OUT / "landsat_candidates.csv", index=False)
print(f"wrote {len(cand)} rows -> {(OUT / 'landsat_candidates.csv').relative_to(config.ROOT)}")

## 3 · One scene per year: May vs November

Rule, applied per calendar year to the scenes acquired in the month:
1. reject if the box is not fully covered or B8 fails the validity check;
2. reject if in-box cloud/shadow (any of QA bits 1-4) is **more than 1 %**;
3. reject as **haze/dust suspected** if the B8 texture index is below half the median texture of the month's other clear, covering scenes from the same sensor family (ETM+ or OLI). `QA_PIXEL` does not flag dust, and dust flattens the image;
4. reject scenes listed in `VISUAL_REJECT`: problems seen by eye in the thumbnails that neither QA nor the haze test caught. Each entry states what was seen. (The list was filled after looking at the first run's thumbnails; it is part of the method, not a hidden edit.);
5. among the survivors, pick the scene closest to a fixed target date (**20 May** or **10 November**) so the sun angle stays as constant as possible across years; ties go to the lower in-box cloud. Survivors not picked are logged as "passed, not picked".

The 16 target years are 2000-2002 and 2013-2025 (handoff §3 Step 1). 2003 is reported as an optional extra: Landsat 7 still worked until 31 May 2003, so a May 2003 scene can exist; no November one can.

In [ ]:
# WHAT THIS DOES: applies the per-year selection rule for May and November and builds a rejection log with the
# reason for every scene not picked.
TARGET_YEARS = [2000, 2001, 2002] + list(range(2013, 2026))
TARGETS = {"May": (5, 20), "November": (11, 10)}
VISUAL_REJECT = {
    "LE07_L1TP_160043_20020509_20200916_02_T1": (
        "visual QA: dark, irregular patches over roughly the SW quarter of the box in B8 and in true colour, "
        "also cooler in the thermal band (cloud shadow or rain-wetted sand); QA_PIXEL flags 0 % (ETM+ CFMask missed it)"),
}

def select(month_name):
    m, d = TARGETS[month_name]
    picks, log = [], []
    for y in sorted(scenes.year.unique()):
        c = scenes[(scenes.year == y) & (scenes.month == m)].copy()
        if c.empty:
            log.append({"month": month_name, "year": y, "scene_id": None, "date": None, "reason": "no scene acquired on this path/row in this month"})
            continue
        c["days_from_target"] = (c.date - pd.Timestamp(y, m, d)).dt.days.abs()
        for _, r in c.iterrows():
            if not r.covers_box:
                why = "does not fully cover the box"
            elif not r.b8_ok:
                why = "B8 invalid/saturated in box"
            elif not r.cloud_ok:
                why = (f"in-box cloud/shadow {r.pct_any_cloud_shadow:.2f} % > {CLOUD_MAX_PCT} % "
                       f"(cloud {r.pct_cloud:.2f}, dilated {r.pct_dilated_cloud:.2f}, cirrus {r.pct_cirrus:.2f}, shadow {r.pct_cloud_shadow:.2f})")
            elif not r.haze_ok:
                why = (f"haze/dust suspected: B8 texture {r.b8_texture:.3f} < {HAZE_FRACTION} x month median "
                       f"{TEX_REF[(m, r.family)]:.3f} ({r.family}); MERRA-2 AOD550 {r.merra2_aod550_total:.2f}")
            elif r.scene_id in VISUAL_REJECT:
                why = VISUAL_REJECT[r.scene_id]
            else:
                why = None
            c.loc[_, "reject"] = why
        ok = c[c.reject.isna()].sort_values(["days_from_target", "pct_any_cloud_shadow"])
        if ok.empty:
            log.append({"month": month_name, "year": y, "scene_id": None, "date": None, "reason": "NO ACCEPTABLE SCENE this year"})
        else:
            picks.append(ok.iloc[0])
        for _, r in c.iterrows():
            if r.reject is not None and isinstance(r.reject, str):
                log.append({"month": month_name, "year": y, "scene_id": r.scene_id, "date": r.date.date(), "reason": r.reject})
            elif not ok.empty and r.scene_id != ok.iloc[0].scene_id:
                log.append({"month": month_name, "year": y, "scene_id": r.scene_id, "date": r.date.date(),
                            "reason": f"passed, not picked ({r.days_from_target} d from target vs {ok.iloc[0].days_from_target} d)"})
    return pd.DataFrame(picks).reset_index(drop=True), pd.DataFrame(log)

picks, logs = {}, {}
for mn in TARGETS:
    picks[mn], logs[mn] = select(mn)
rejections = pd.concat(logs.values(), ignore_index=True)
rejections.to_csv(OUT / "scene_rejections.csv", index=False)
print(f"rejection log: {len(rejections)} entries -> {(OUT / 'scene_rejections.csv').relative_to(config.ROOT)}")
for mn in TARGETS:
    print(f"\n--- {mn}: log ---")
    print(logs[mn].to_string(index=False))

In [ ]:
# WHAT THIS DOES: shows the per-year picks for both months side by side with in-box cloud, sun angles, texture
# and the MERRA-2 haze indicator, and lists which of the 16 target years each month covers.
show = ["year", "scene_id", "date", "pct_any_cloud_shadow", "pct_cloud", "pct_cloud_shadow", "cloud_cover_scene",
        "sun_elevation", "sun_azimuth", "b8_texture", "merra2_aod550_total", "merra2_aod550_dust"]
for mn in TARGETS:
    p = picks[mn]
    print(f"\n=== {mn} picks ({len(p)} years) ===")
    print(p[show].assign(date=p.date.dt.date).round(3).to_string(index=False))
    missing = [y for y in TARGET_YEARS if y not in set(p.year)]
    print(f"target years covered: {sum(y in set(p.year) for y in TARGET_YEARS)}/16; missing: {missing}; extra years: {sorted(set(p.year) - set(TARGET_YEARS))}")

# Off-month fill-in options for missing target years: the clean scene nearest the target date within +-35 days,
# outside the month. NOT added to the lists; offered for the user to accept or refuse.
FILL_INS = {}
ok_all = scenes[scenes.covers_box & scenes.b8_ok & scenes.cloud_ok & scenes.haze_ok & ~scenes.scene_id.isin(VISUAL_REJECT)]
for mn, (m, d) in TARGETS.items():
    FILL_INS[mn] = []
    for y in [y for y in TARGET_YEARS if y not in set(picks[mn].year)]:
        c = ok_all[(ok_all.year == y) & (ok_all.month != m)].copy()
        c["days"] = (c.date - pd.Timestamp(y, m, d)).dt.days.abs()
        c = c[c.days <= 35].sort_values("days")
        for _, r in c.iterrows():
            FILL_INS[mn].append({"year": y, "scene_id": r.scene_id, "date": r.date.strftime("%Y-%m-%d"), "days_from_target": int(r.days),
                                 "sun_elevation_deg": round(float(r.sun_elevation), 2), "sun_azimuth_deg": round(float(r.sun_azimuth), 2),
                                 "inbox_cloud_shadow_pct": round(float(r.pct_any_cloud_shadow), 3), "b8_texture": round(float(r.b8_texture), 4)})
        if c.empty:
            FILL_INS[mn].append({"year": y, "scene_id": None, "note": "no clean scene within 35 days of the target date"})
    print(f"\n{mn} off-month fill-in options (not used unless confirmed):")
    for f in FILL_INS[mn]:
        print("  ", f)

In [ ]:
# WHAT THIS DOES: fetches small B8 thumbnails (box only, ~28 m display pixels, stretched per scene from the in-box
# 2nd-98th percentile) of every pick from Earth Engine and plots May and November side by side for each year.
# Display only; no measurement uses these images.
def thumb(r):
    url = ee.Image(f"{r.collection}/{r.ee_id}").select("B8").getThumbURL(
        {"region": BOX, "dimensions": 360, "crs": f"EPSG:{EPSG}", "min": r.b8_p2, "max": r.b8_p98, "format": "png"})
    for attempt in range(4):
        try:
            resp = requests.get(url, timeout=120); resp.raise_for_status()
            return plt.imread(io.BytesIO(resp.content), format="png")
        except Exception:
            if attempt == 3:
                raise
            time.sleep(3)

all_years = sorted(set(picks["May"].year) | set(picks["November"].year))
jobs = [(mn, r) for mn in TARGETS for _, r in picks[mn].iterrows()]
with ThreadPoolExecutor(8) as ex:
    imgs = dict(zip([(mn, r.year) for mn, r in jobs], ex.map(lambda j: thumb(j[1]), jobs)))

ncol = 6
nrow = math.ceil(len(all_years) / (ncol // 2))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 2.6, nrow * 2.75))
for k, y in enumerate(all_years):
    for c, mn in enumerate(TARGETS):
        ax = axes.flat[2 * k + c]
        ax.set_xticks([]); ax.set_yticks([])
        if (mn, y) in imgs:
            r = picks[mn][picks[mn].year == y].iloc[0]
            ax.imshow(imgs[(mn, y)], cmap="gray")
            ax.set_title(f"{r.date.date()}  L{r.sensor.split()[1]}\nsun el {r.sun_elevation:.0f}°  tex {r.b8_texture:.3f}", fontsize=7.5)
        else:
            ax.set_facecolor("#dddddd"); ax.set_title(f"{y} {mn[:3]}: no scene", fontsize=7.5)
for ax in axes.flat[2 * len(all_years):]:
    ax.axis("off")
fig.suptitle(f"{SITE['name']}: Landsat B8 (15 m pan) picks, May (left) vs November (right) of each year\n"
             f"box W {BBOX[0]} S {BBOX[1]} E {BBOX[2]} N {BBOX[3]}, north up, display stretch per scene", fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.965])
fig.savefig(OUT / "b8_thumbnails_may_vs_nov.png", dpi=110)
plt.show()

In [ ]:
# WHAT THIS DOES: zooms into the same ~3 x 3 km patch of the dune belt east of the race track (centre chosen by eye
# from the thumbnails) for one early and one recent year at full 15 m display resolution, so the difference in dune
# relief between May and November is visible.
ZOOM_LAT, ZOOM_LON = 24.058, 55.557
def crop(r, size_deg=0.0135):
    reg = ee.Geometry.Rectangle([ZOOM_LON - size_deg, ZOOM_LAT - size_deg, ZOOM_LON + size_deg, ZOOM_LAT + size_deg], "EPSG:4326", False)
    url = ee.Image(f"{r.collection}/{r.ee_id}").select("B8").getThumbURL(
        {"region": reg, "dimensions": 200, "crs": f"EPSG:{EPSG}", "min": r.b8_p2, "max": r.b8_p98, "format": "png"})
    return plt.imread(io.BytesIO(requests.get(url, timeout=120).content), format="png")

zoom_years = [y for y in (2001, 2024) if all(y in set(picks[mn].year) for mn in TARGETS)]
fig, axes = plt.subplots(1, 2 * len(zoom_years), figsize=(4.2 * len(zoom_years) * 2, 4.4))
for k, (y, mn) in enumerate([(y, mn) for y in zoom_years for mn in TARGETS]):
    r = picks[mn][picks[mn].year == y].iloc[0]
    axes[k].imshow(crop(r), cmap="gray"); axes[k].set_xticks([]); axes[k].set_yticks([])
    axes[k].set_title(f"{r.date.date()} ({r.sensor}), sun el {r.sun_elevation:.0f}° az {r.sun_azimuth:.0f}°", fontsize=8)
fig.suptitle(f"~3 x 3 km centred {ZOOM_LAT} N {ZOOM_LON} E (dune belt E of the race track), 15 m pixels shown ~1:1", fontsize=10)
fig.tight_layout(); fig.savefig(OUT / "b8_zoom_may_vs_nov.png", dpi=110); plt.show()

In [ ]:
# WHAT THIS DOES: computes the comparison numbers behind the month choice: years covered, spread of sun elevation
# and azimuth across the picks, texture (all picks and per sensor), and haze (MERRA-2 AOD at acquisition) for the
# picks and for all candidate scenes in each month.
rows = []
for mn, (m, _) in TARGETS.items():
    p = picks[mn]
    allm = scenes[scenes.month == m]
    rows.append({
        "month": mn,
        "target_years_covered": f"{sum(y in set(p.year) for y in TARGET_YEARS)}/16",
        "sun_el_min": p.sun_elevation.min(), "sun_el_max": p.sun_elevation.max(), "sun_el_range": np.ptp(p.sun_elevation),
        "sun_az_range": np.ptp(p.sun_azimuth),
        "texture_median_all": p.b8_texture.median(),
        "texture_median_L7": p[p.sensor == "Landsat 7 ETM+"].b8_texture.median(),
        "texture_median_OLI": p[p.sensor != "Landsat 7 ETM+"].b8_texture.median(),
        "aod_total_median_picks": p.merra2_aod550_total.median(), "aod_total_max_picks": p.merra2_aod550_total.max(),
        "aod_dust_median_picks": p.merra2_aod550_dust.median(),
        "aod_total_median_all_scenes_in_month": allm.merra2_aod550_total.median(),
        "share_scenes_in_month_passing_1pct": allm.cloud_ok.mean(),
    })
comparison = pd.DataFrame(rows).set_index("month").T
print(comparison.round(3).to_string())

# Texture difference within the same year and sensor, November minus May (paired):
pair = picks["May"][["year", "b8_texture"]].merge(picks["November"][["year", "b8_texture"]], on="year", suffixes=("_may", "_nov"))
pair["nov_over_may"] = pair.b8_texture_nov / pair.b8_texture_may
print(f"\npaired years: {len(pair)}; November/May texture ratio median {pair.nov_over_may.median():.2f}, "
      f"min {pair.nov_over_may.min():.2f}, max {pair.nov_over_may.max():.2f}")

## Recommendation: November (a proposal; the user confirms the list)

These are the numbers from the cells above for this box (WRS-2 160/043, Collection 2 Tier 1):

| | May (target 20 May) | November (target 10 Nov) |
|---|---|---|
| Target years with an accepted scene | **15/16**: 2002 is lost because its only May scene (2002-05-09) fails visual QA. 2003-05-28 is an optional extra. | **14/16**: 2002 has no Landsat 7 acquisition in November. 2018's two scenes are cloudy inside the box (9.9 %, mostly cirrus, and 41 %). |
| Sun elevation across the picks | 66.5-69.0° (spread 2.6°) | 38.6-46.5° (spread 7.9°). The two ETM+ scenes are late November (38.6-39.3°); OLI is 42.1-46.5°. |
| Sun azimuth spread | 8.8° | 2.9° |
| Texture index, median of picks | ETM+ 0.030, OLI 0.051 | ETM+ 0.070, OLI 0.072 |
| Same-year texture, Nov / May | | median **1.52×** (range 1.19-2.27). November is higher in all 14 paired years. |
| MERRA-2 AOD 550 nm at acquisition, picks | median 0.28 (dust 0.22), max 0.66 | median 0.19 (dust 0.11), max 0.51 |
| Rejected as hazy | 2018-05-13 (AOD 1.29, texture 0.010) | none |

**Why November:**
1. **Relief and texture for feature tracking.** Phase correlation needs 15 m-scale contrast. November's low sun casts slip-face shading that May's near-overhead sun does not. Its texture index is about 1.5× higher every year, and 2.3× higher for the Landsat 7 scenes that anchor the 25-year baseline (0.070 vs 0.030). The May ETM+ scenes have the weakest texture in the whole set.
2. **Less dust.** November's MERRA-2 aerosol optical depth at acquisition is about two-thirds of May's, and its dust share is half. A May pick (2018-05-13) had to be rejected for dust that `QA_PIXEL` did not flag.
3. **The baseline is intact.** It runs from 2000-11-27 to 2025-11-08 (24.95 years), with 2001-11-30 as a second early epoch. Losing 2002 and 2018 removes one early epoch and turns 2017→2019 into a two-year pair, which is fine for yearly-rate estimates.

**What it costs, and how notebook 02 has to handle it:**
- **Sun-angle spread is larger** (7.9° vs 2.6°), mostly between the late-November ETM+ scenes and the mid-November OLI scenes. Shadow length scales with cot(elevation): about 1.25 at 38.6° and 1.0 at 45°, so the ETM+ shadows are about 25 % longer. A shading edge can shift slightly with sun angle even when the dune does not move. Because the azimuth is nearly constant (2.9° spread), any such bias has a consistent direction. Notebook 02 must measure it on stable ground and in the residuals of same-sensor OLI pairs, and report it.
- **Two years are missing.** For 2018, the cells above list off-month options; the cleanest close one is 2018-12-07 (0 % in-box cloud, sun 38.6°, inside November's elevation range). It is **not** in the list unless you accept it.

**May remains a valid alternative.** It has one more year and a more uniform sun, but weaker relief and more dust. Both lists are written to `scenes_selected.json`.

**Things this step found that differ from the Phase 1 notes** (which used scene-level STAC cloud):
- November 2018: the scene STAC rated at 0.32 % cloud has **9.9 %** cloud/cirrus/shadow inside the box. November therefore covers 14/16 years, not 15/16.
- May 2002: `QA_PIXEL` (ETM+ CFMask) flags 0 %, but the scene has dark, cool patches over the SW of the box. It is rejected by visual QA, so May covers 15/16 years, not 16/16.
- The scene-level pick for May 2018 (2018-05-29 in Phase 1) is what this rule also picks. The closer-to-target 2018-05-13 is rejected for dust.
- Earth Engine has 5 Landsat 9 scenes that the STAC query did not return. None of them changes a pick: the two from November 2023 are rejected for cloud (2023-11-11, 16.9 %) or lie further from the target date (2023-11-27).
- **Surface change:** the area of the race track looks very different before 2003 (a large dark flat) and from 2013 onwards (the built track and a settlement). Tracking across 2000→2025 there will measure construction, not sand. Notebook 02 has to mask it.

In [ ]:
# WHAT THIS DOES: writes pipeline/outputs/<site>/scenes_selected.json with the recommended month's list and the
# alternative month's list, marked as a proposal awaiting the user's confirmation.
RECOMMENDED, ALTERNATIVE = "November", "May"

def as_list(p):
    out = []
    for _, r in p.iterrows():
        out.append({"year": int(r.year), "date": r.date.strftime("%Y-%m-%d"), "scene_id": r.scene_id,
                    "ee_asset": f"{r.collection}/{r.ee_id}", "sensor": f"{r.sensor} B8", "pixel_m": 15,
                    "sun_elevation_deg": round(float(r.sun_elevation), 2), "sun_azimuth_deg": round(float(r.sun_azimuth), 2),
                    "cloud_cover_scene_pct": round(float(r.cloud_cover_scene), 2),
                    "inbox_cloud_shadow_pct": round(float(r.pct_any_cloud_shadow), 3),
                    "b8_texture": round(float(r.b8_texture), 4),
                    "merra2_aod550_total": round(float(r.merra2_aod550_total), 3),
                    "in_16_target_years": int(r.year) in TARGET_YEARS})
    return out

sel = {
    "status": "proposed, awaiting user confirmation",
    "site": SID, "site_name": SITE["name"], "bbox": SITE["bbox"],
    "wrs_path": SITE["wrs_path"], "wrs_row": SITE["wrs_row"],
    "generated_utc": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "rule": (f"Per calendar year, scenes in the month on WRS-2 {SITE['wrs_path']}/{SITE['wrs_row']:03d} (Collection 2 Tier 1) "
             f"that fully cover the box with valid B8; reject if in-box cloud/dilated cloud/cirrus/shadow "
             f"(QA_PIXEL bits 1-4) > {CLOUD_MAX_PCT} %; pick the survivor closest to the target date "
             "(20 May / 10 Nov). Landsat 7 only before 2003-05-31 (SLC failure)."),
    "visual_reject": VISUAL_REJECT,
    "haze_rule": f"reject if B8 texture < {HAZE_FRACTION} x median texture of the month's clear covering scenes of the same sensor family",
    "recommended_month": RECOMMENDED,
    "recommended": as_list(picks[RECOMMENDED]),
    "alternative_month": ALTERNATIVE,
    "alternative": as_list(picks[ALTERNATIVE]),
    "off_month_fill_in_options": FILL_INS,
    "comparison": {k: {kk: (round(float(vv), 4) if isinstance(vv, (float, np.floating)) else vv) for kk, vv in v.items()}
                   for k, v in comparison.to_dict().items()},
    "files": {"candidates": "landsat_candidates.csv", "rejections": "scene_rejections.csv",
              "thumbnails": "b8_thumbnails_may_vs_nov.png", "zoom": "b8_zoom_may_vs_nov.png"},
    "note": "Nothing has been downloaded at full resolution. Notebook 02 uses the confirmed list.",
}
(OUT / "scenes_selected.json").write_text(json.dumps(sel, indent=1) + "\n")
print(f"wrote {(OUT / 'scenes_selected.json').relative_to(config.ROOT)}: {RECOMMENDED} {len(sel['recommended'])} scenes (recommended), "
      f"{ALTERNATIVE} {len(sel['alternative'])} scenes (alternative)")
for s in sel["recommended"]:
    print(f"  {s['year']}  {s['date']}  {s['scene_id']}")

## 4 · Wind: Fryberger drift potential from MERRA-2

**Method** (Fryberger & Dean 1979, "Dune forms and wind regime", USGS Prof. Paper 1052, ch. F):
- V = hourly-mean wind speed at 10 m in **knots** (`U10M`, `V10M` in m/s × 1/0.514444); threshold **Vt = 12 kt**.
- **DP = Σ V²(V − Vt)/100 · t**, summed over hours with V > Vt, where **t is the percentage of the period that hour represents**. With hourly data each hour is its own class, so for a year t = 100/N (N = valid hours in that year), and DP_year = mean over all hours of V²(V − Vt)₊. DP is in vector units (VU) per year, which is what the class limits refer to: **low < 200 ≤ intermediate < 400 ≤ high**. (The handoff's shorthand "DP = U²(U − Ut)" leaves out the /100 · t normalisation; without it the numbers scale with record length and the class limits mean nothing.)
- **FROM direction** (meteorological) = atan2(−U, −V); sand moves **downwind** = FROM + 180°.
- Each hour contributes a vector of length V²(V − Vt)/100 · t pointing downwind. **RDP** = length of the vector sum, **RDD** = its bearing (direction sand moves *to*), **RDP/DP** = directional variability (near 1 = one wind direction; small = opposing winds).
- Whole period: DP = mean annual DP; RDP/RDD from the mean of the yearly resultant vectors (each year weighted equally); sand rose = mean annual DP per 16 sectors (22.5°, sector centred on its bearing), binned by **FROM** direction, the usual Fryberger sand-rose convention.
- Missing hours are dropped and t uses the valid count (the valid hours represent the year); the count is reported.

The maths lives in `pipeline/dunesight/wind.py`; the next cell tests it on cases that can be checked by hand.

In [ ]:
# WHAT THIS DOES: unit-tests the direction and DP maths on synthetic winds whose answers can be worked out by hand.
KT = 0.514444  # m/s per knot
def ms(kt): return kt * KT
def close(a, b, tol=1e-6): return abs(a - b) <= tol * max(1.0, abs(b))
N = 8760
tests = []

# 1. A pure northerly (blowing FROM north, so U = 0, V < 0): FROM 0, downwind 180.
tests.append(("northerly FROM = 0", close(float(wind.from_dir_deg(0.0, -5.0)), 0.0)))
tests.append(("northerly downwind = 180", close(float(wind.downwind_deg(0.0, -5.0)), 180.0)))
# 2. Easterly (U < 0): FROM 90, downwind 270. Westerly (U > 0): FROM 270.
tests.append(("easterly FROM = 90", close(float(wind.from_dir_deg(-5.0, 0.0)), 90.0)))
tests.append(("westerly FROM = 270", close(float(wind.from_dir_deg(5.0, 0.0)), 270.0)))
# 3. A NW shamal (blowing toward SE: U > 0, V < 0): FROM 315, downwind 135.
tests.append(("NW wind FROM = 315", close(float(wind.from_dir_deg(1.0, -1.0)), 315.0)))
tests.append(("NW wind downwind = 135", close(float(wind.downwind_deg(1.0, -1.0)), 135.0)))
# 4. 20 kt = 10.28888 m/s converts back to 20 kt.
tests.append(("20 kt round trip", close(float(wind.speed_kt(ms(20), 0)), 20.0)))
# 5. Constant 20 kt northerly all year: DP = 20^2 * (20-12) / 100 * 100 = 3200 VU, RDP = DP, RDD = 180, RDP/DP = 1.
r = wind.fryberger(np.zeros(N), np.full(N, -ms(20)))
tests.append(("constant 20 kt: DP = 3200", close(r["DP"], 3200.0)))
tests.append(("constant 20 kt: RDP = 3200, RDD = 180", close(r["RDP"], 3200.0) and close(r["RDD_deg"], 180.0)))
tests.append(("constant 20 kt: rose all in FROM sector 0", close(r["rose_from"][0], 3200.0) and close(r["rose_from"][1:].sum(), 0.0)))
# 6. Below threshold (10 kt) and exactly at threshold (12 kt): DP = 0.
tests.append(("10 kt: DP = 0", wind.fryberger(np.zeros(N), np.full(N, ms(10)))["DP"] == 0.0))
tests.append(("12 kt: DP = 0", close(wind.fryberger(np.zeros(N), np.full(N, ms(12)))["DP"], 0.0, 1e-9)))
# 7. 25 % of hours a 20 kt westerly, rest calm: DP = 3200 * 0.25 = 800, sand moves east (RDD 90).
u = np.zeros(N); u[: N // 4] = ms(20)
r = wind.fryberger(u, np.zeros(N))
tests.append(("25 % westerly 20 kt: DP = 800, RDD = 90", close(r["DP"], 800.0) and close(r["RDD_deg"], 90.0)))
# 8. Half the year from N, half from S at 20 kt: DP = 3200, RDP = 0 (opposing winds cancel).
v = np.where(np.arange(N) < N // 2, -ms(20), ms(20))
r = wind.fryberger(np.zeros(N), v)
tests.append(("opposing N/S: DP = 3200, RDP = 0", close(r["DP"], 3200.0) and r["RDP"] < 1e-6))
# 9. Half from N, half from E at 20 kt: each 1600 VU; resultant 1600*sqrt(2) = 2262.74 toward 225 (SW).
u = np.where(np.arange(N) < N // 2, 0.0, -ms(20)); v = np.where(np.arange(N) < N // 2, -ms(20), 0.0)
r = wind.fryberger(u, v)
tests.append(("N + E: RDP = 2262.74, RDD = 225", close(r["RDP"], 1600 * math.sqrt(2)) and close(r["RDD_deg"], 225.0)))
# 10. Missing hours (NaN) are dropped and t uses the valid count: same DP as test 5.
v = np.full(N, -ms(20)); v[:100] = np.nan
tests.append(("missing hours ignored", close(wind.fryberger(np.zeros(N), v)["DP"], 3200.0) and wind.fryberger(np.zeros(N), v)["n_valid"] == N - 100))
# 11. Sector binning: FROM 350 and 11.2 -> sector 0; 11.3 -> sector 1 (22.5); 348.7 -> sector 15 (337.5).
tests.append(("sector binning", list(wind.sector_index([350, 11.2, 11.3, 348.7])) == [0, 0, 1, 15]))
# 12. DP class limits.
tests.append(("DP classes", [wind.dp_class(x) for x in (199.9, 200, 399.9, 400)] == ["low", "intermediate", "intermediate", "high"]))
# 13. One hour at 30 kt in a 100-hour record: DP = 900*18/100 * (100/100) = 162.
v = np.zeros(100); v[0] = -ms(30)
tests.append(("one 30 kt hour in 100: DP = 162", close(wind.fryberger(np.zeros(100), v)["DP"], 162.0)))

for name, ok in tests:
    print(("PASS " if ok else "FAIL ") + name)
assert all(ok for _, ok in tests), "wind maths unit test failed"
print(f"all {len(tests)} synthetic tests passed")

### Which MERRA-2 grid cell?

MERRA-2's native grid has cell centres at latitude −90 + 0.5 j and longitude −180 + 0.625 i. Earth Engine's `NASA/GSFC/MERRA/slv/2` declares a pixel grid whose origin is (−180.5, −90.5), which puts every pixel **0.1875° west and 0.25° south** of the native cell with the same index. The next cell checks this against NASA POWER (which serves the native MERRA-2 cells) for one hour, and then samples the Earth Engine pixel that holds the native cell containing the box centroid, instead of the pixel Earth Engine would pick at the centroid's coordinates.

In [ ]:
# WHAT THIS DOES: reads Earth Engine's MERRA-2 pixel grid, works out the native cell containing the box centroid,
# and verifies for one hour which Earth Engine pixel holds that cell by comparing with NASA POWER at the centroid.
slv = ee.ImageCollection("NASA/GSFC/MERRA/slv/2")
proj = slv.first().select("U10M").projection().getInfo()
T = proj["transform"]
cell = merra2.native_cell(LAT_C, LON_C)
naive_ij = merra2.ee_pixel_at(T, LAT_C, LON_C)
print("Earth Engine MERRA-2 projection:", proj["crs"], T)
print(f"native MERRA-2 cell containing the centroid: i={cell['i']} j={cell['j']}, centre {cell['lat_c']} N {cell['lon_c']} E, "
      f"spans {cell['south']}-{cell['north']} N, {cell['west']}-{cell['east']} E (0.5 deg x 0.625 deg, about "
      f"{0.5 * 110.6:.0f} km x {0.625 * 111.32 * math.cos(math.radians(cell['lat_c'])):.0f} km)")
print(f"pixel Earth Engine would sample at the centroid coordinates: i={naive_ij[0]} j={naive_ij[1]}")

UA = {"User-Agent": "DuneSight-pipeline/0.1 (open-source research)"}
def power_hourly(year_start, year_end, lat=LAT_C, lon=LON_C):
    url = ("https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=U10M,V10M&community=RE"
           f"&longitude={lon}&latitude={lat}&start={year_start}&end={year_end}&format=JSON&time-standard=UTC")
    for attempt in range(5):
        try:
            resp = requests.get(url, headers=UA, timeout=300); resp.raise_for_status()
            p = resp.json()["properties"]["parameter"]
            df = pd.DataFrame({"U10M": p["U10M"], "V10M": p["V10M"]})
            df.index = pd.to_datetime(df.index, format="%Y%m%d%H", utc=True)
            return df.replace(-999.0, np.nan)
        except Exception:
            if attempt == 4:
                raise
            time.sleep(5 * (attempt + 1))

check_hour = "2020-01-01T06:00:00"
pw = power_hourly("20200101", "20200101").loc[pd.Timestamp(check_hour, tz="UTC")]
img = slv.filterDate(check_hour, ee.Date(check_hour).advance(1, "hour")).first().select(["U10M", "V10M"])
cands = {"EE pixel at native-cell index (used)": (cell["i"], cell["j"]), "EE pixel at centroid coordinates": naive_ij}
for di in (-1, 0, 1):
    for dj in (-1, 0, 1):
        cands.setdefault(f"EE pixel ({cell['i'] + di},{cell['j'] + dj})", (cell["i"] + di, cell["j"] + dj))
print(f"\nhour {check_hour} UTC; NASA POWER at the centroid: U10M={pw.U10M:.2f} V10M={pw.V10M:.2f} m/s")
match = {}
for name, (i, j) in cands.items():
    x, y = merra2.ee_pixel_centre(T, i, j)
    v = img.reduceRegion(ee.Reducer.first(), ee.Geometry.Point([x, y]), crs="EPSG:4326", crsTransform=T).getInfo()
    match[name] = abs(v["U10M"] - pw.U10M) < 0.006 and abs(v["V10M"] - pw.V10M) < 0.006
    print(f"  {name:40s} i={i} j={j}  U10M={v['U10M']:.3f} V10M={v['V10M']:.3f}  matches POWER: {match[name]}")
assert match["EE pixel at native-cell index (used)"], "native-cell pixel does not match POWER; grid assumption wrong"
SAMPLE_XY = merra2.ee_pixel_centre(T, cell["i"], cell["j"])
print(f"\nsampling Earth Engine at {SAMPLE_XY} (EE coordinates), which holds native cell centre {cell['lat_c']} N {cell['lon_c']} E")

In [ ]:
# WHAT THIS DOES: pulls hourly U10M/V10M for 2000-01-01 to 2024-12-31 from Earth Engine for that cell (one request
# per year, in parallel), caches it in pipeline/raw/<site>/, and reports missing or duplicate hours.
WIND_CACHE = RAW / f"merra2_u10v10_hourly_i{cell['i']}_j{cell['j']}.csv"
YEARS = list(range(2000, 2025))

def merra_year(y, xy=SAMPLE_XY):
    c = slv.filterDate(f"{y}-01-01", f"{y + 1}-01-01").select(["U10M", "V10M"])
    for attempt in range(5):
        try:
            rows = c.getRegion(ee.Geometry.Point(list(xy)), scale=None, crs="EPSG:4326", crsTransform=T).getInfo()
            df = pd.DataFrame(rows[1:], columns=rows[0])
            df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True)
            return df[["time", "U10M", "V10M"]]
        except Exception:
            if attempt == 4:
                raise
            time.sleep(5 * (attempt + 1))

if WIND_CACHE.exists() and not REFRESH:
    hourly = pd.read_csv(WIND_CACHE, parse_dates=["time"])
    print(f"loaded cached MERRA-2 series: {len(hourly)} rows ({WIND_CACHE.relative_to(config.ROOT)})")
else:
    t0 = time.time()
    with ThreadPoolExecutor(6) as ex:
        hourly = pd.concat(list(ex.map(merra_year, YEARS)), ignore_index=True)
    hourly.to_csv(WIND_CACHE, index=False)
    print(f"queried Earth Engine: {len(hourly)} rows in {time.time() - t0:.0f} s")

hourly["time"] = pd.to_datetime(hourly["time"], utc=True)
dups = int(hourly.time.duplicated().sum())
hourly = hourly.drop_duplicates("time").set_index("time").sort_index()
full = pd.date_range("2000-01-01T00:00Z", "2024-12-31T23:00Z", freq="h")
hourly = hourly.reindex(full)
miss = hourly.U10M.isna() | hourly.V10M.isna()
print(f"expected hours {len(full)}, present {int((~miss).sum())}, missing {int(miss.sum())}, duplicates dropped {dups}")
if miss.any():
    print(hourly[miss].groupby(hourly[miss].index.year).size().rename("missing_hours"))
print("speed (m/s):", (np.hypot(hourly.U10M, hourly.V10M)).describe().round(2).to_dict())

In [ ]:
# WHAT THIS DOES: computes yearly and whole-period DP, RDP, RDD and RDP/DP, the DP class and the 16-sector sand rose
# from the hourly MERRA-2 series.
ok = ~miss
yr_rows, period = wind.fryberger_by_year(hourly.index.year[ok], hourly.U10M[ok].values, hourly.V10M[ok].values)
yearly = pd.DataFrame([{k: r[k] for k in ("year", "DP", "RDP", "RDD_deg", "RDP_DP", "pct_hours_above_vt", "n_valid", "n_missing")}
                       for r in yr_rows])
yearly["dp_class"] = yearly.DP.map(wind.dp_class)
print(yearly.round(2).to_string(index=False))
print(f"\nWhole period 2000-2024 (mean annual): DP = {period['DP']:.1f} VU ({period['dp_class']}), RDP = {period['RDP']:.1f} VU, "
      f"RDD = {period['RDD_deg']:.1f} deg, RDP/DP = {period['RDP_DP']:.3f}")
print(f"yearly DP range {yearly.DP.min():.1f}-{yearly.DP.max():.1f} VU (median {yearly.DP.median():.1f}); "
      f"yearly RDD range {yearly.RDD_deg.min():.1f}-{yearly.RDD_deg.max():.1f} deg; "
      f"hours above 12 kt: {yearly.pct_hours_above_vt.mean():.2f} % on average")
rose = pd.DataFrame({"sector_from_deg": np.arange(16) * 22.5, "dp": period["rose_from"]})
rose["share_pct"] = 100 * rose.dp / rose.dp.sum()
print("\nsand rose (mean annual DP by FROM sector):")
print(rose.round(2).to_string(index=False))
yearly.round(3).to_csv(OUT / "wind_yearly_merra2.csv", index=False)
rose.round(3).to_csv(OUT / "sand_rose_merra2.csv", index=False)

In [ ]:
# WHAT THIS DOES: draws the sand rose (bars point to the FROM direction, length = mean annual DP) with the resultant
# drift arrow (toward RDD), and the yearly DP and RDD series.
fig = plt.figure(figsize=(12, 4.2))
ax = fig.add_subplot(1, 3, 1, projection="polar")
ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
ax.bar(np.radians(rose.sector_from_deg), rose.dp, width=np.radians(20), color="#4a7fb5", edgecolor="white", linewidth=1)
ax.annotate("", xy=(np.radians(period["RDD_deg"]), rose.dp.max()), xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color="#c0392b", lw=2))
ax.set_title(f"Sand rose (FROM), DP {period['DP']:.0f} VU/yr\nred arrow: RDD {period['RDD_deg']:.0f}° (sand moves to)", fontsize=9)
ax.tick_params(labelsize=7)
ax2 = fig.add_subplot(1, 3, 2)
ax2.bar(yearly.year, yearly.DP, color="#4a7fb5"); ax2.bar(yearly.year, yearly.RDP, color="#9cc3e6", width=0.4)
ax2.set_title("Yearly DP (wide) and RDP (narrow), VU", fontsize=9); ax2.tick_params(labelsize=7)
ax3 = fig.add_subplot(1, 3, 3)
ax3.plot(yearly.year, yearly.RDD_deg, "o-", color="#4a7fb5", lw=2, ms=4)
ax3.set_ylim(0, 360); ax3.set_yticks(range(0, 361, 45)); ax3.set_title("Yearly RDD (deg, direction sand moves to)", fontsize=9)
ax3.tick_params(labelsize=7); ax3.grid(alpha=0.3)
fig.suptitle(f"MERRA-2 hourly 10 m wind, cell centred {cell['lat_c']} N {cell['lon_c']} E, 2000-2024", fontsize=10)
fig.tight_layout(); fig.savefig(OUT / "wind_sand_rose_merra2.png", dpi=110); plt.show()

In [ ]:
# WHAT THIS DOES: checks whether the drift direction is physically plausible for the UAE (NW shamal -> sand toward
# the SE) by breaking DP and RDD down by month and by time of day.
h = hourly[ok].copy()
h["kt"] = wind.speed_kt(h.U10M, h.V10M)
h["from"] = wind.from_dir_deg(h.U10M, h.V10M)
def grp_stats(g):
    r = wind.fryberger(g.U10M.values, g.V10M.values)
    # scale so the numbers are this group's contribution to the mean annual DP
    share = len(g) / len(h) * len(YEARS)
    return pd.Series({"DP_contrib": r["DP"] * share, "RDD_deg": r["RDD_deg"], "RDP_DP": r["RDP_DP"],
                      "pct_hours_above_12kt": r["pct_hours_above_vt"], "mean_kt": g.kt.mean()})
by_month = h.groupby(h.index.month).apply(grp_stats)
by_month.index.name = "month"
print("contribution to mean annual DP by calendar month (VU/yr), with that month's RDD:")
print(by_month.round(2).to_string())
by_hour = h.groupby(h.index.hour).apply(grp_stats)
by_hour.index.name = "hour_utc"
print("\nby hour of day (UTC; local time = UTC+4):")
print(by_hour.round(2).to_string())
strong = h[h.kt > 12]
print(f"\nhours above 12 kt: {len(strong)} ({100 * len(strong) / len(h):.2f} %); FROM-direction quartiles of those hours:",
      np.percentile(strong["from"], [25, 50, 75]).round(0))
print("share of above-threshold hours by FROM sector (%):")
print((strong.groupby(wind.sector_index(strong["from"])).size() / len(strong) * 100).rename(lambda k: k * 22.5).round(1).to_string())

### Cross-check with NASA POWER

NASA POWER's hourly meteorology is derived from MERRA-2 on the same native grid, so this checks **our extraction and processing** (grid cell, time stamps, units, the DP code), not the physics. POWER hourly starts in 2001, so the comparison covers 2001-2024.

In [ ]:
# WHAT THIS DOES: downloads NASA POWER hourly U10M/V10M at the box centroid for 2001-2024 (one request per year,
# cached), compares it hour by hour with the Earth Engine series, and compares yearly DP and RDD.
POWER_CACHE = RAW / "power_u10v10_hourly_centroid.csv"
P_YEARS = list(range(2001, 2025))
if POWER_CACHE.exists() and not REFRESH:
    power = pd.read_csv(POWER_CACHE, index_col=0, parse_dates=True)
    print(f"loaded cached POWER series: {len(power)} rows")
else:
    with ThreadPoolExecutor(4) as ex:
        power = pd.concat(list(ex.map(lambda y: power_hourly(f"{y}0101", f"{y}1231"), P_YEARS)))
    power.to_csv(POWER_CACHE)
    print(f"downloaded POWER: {len(power)} rows")
power.index = pd.to_datetime(power.index, utc=True)
pmiss = power.U10M.isna() | power.V10M.isna()
print(f"POWER hours {len(power)}, missing {int(pmiss.sum())}")

j = hourly.join(power.add_suffix("_power"), how="inner").dropna()
du, dv = j.U10M - j.U10M_power, j.V10M - j.V10M_power
worst = (j.V10M - j.V10M_power).abs().idxmax()
print(f"largest single-hour difference: {worst} UTC; EE U,V = {j.U10M[worst]:.2f}, {j.V10M[worst]:.2f}; POWER U,V = {j.U10M_power[worst]:.2f}, {j.V10M_power[worst]:.2f}")
print(f"hours differing by more than 0.01 m/s (beyond POWER's rounding): U {int((du.abs() > 0.0101).sum())}, V {int((dv.abs() > 0.0101).sum())}")
print(f"hourly comparison over {len(j)} common hours: |dU| mean {du.abs().mean():.4f} max {du.abs().max():.3f} m/s; "
      f"|dV| mean {dv.abs().mean():.4f} max {dv.abs().max():.3f} m/s; r(U) {np.corrcoef(j.U10M, j.U10M_power)[0, 1]:.5f}, "
      f"r(V) {np.corrcoef(j.V10M, j.V10M_power)[0, 1]:.5f}  (POWER rounds to 0.01 m/s)")

pok = ~pmiss
p_rows, p_period = wind.fryberger_by_year(power.index.year[pok], power.U10M[pok].values, power.V10M[pok].values)
cmp = pd.DataFrame([{"year": a["year"], "DP_merra2_ee": a["DP"], "DP_power": b["DP"], "RDD_merra2_ee": a["RDD_deg"],
                     "RDD_power": b["RDD_deg"]} for a, b in zip([r for r in yr_rows if r["year"] >= 2001], p_rows)])
cmp["DP_diff_pct"] = 100 * (cmp.DP_power - cmp.DP_merra2_ee) / cmp.DP_merra2_ee
cmp["RDD_diff_deg"] = wind.angle_diff(cmp.RDD_power, cmp.RDD_merra2_ee)
print(cmp.round(2).to_string(index=False))
ee_2001 = wind.fryberger_by_year(hourly.index.year[ok & (hourly.index.year >= 2001)],
                                 hourly.U10M[ok & (hourly.index.year >= 2001)].values, hourly.V10M[ok & (hourly.index.year >= 2001)].values)[1]
POWER_AGREEMENT = {
    "period": "2001-2024", "common_hours": int(len(j)),
    "DP_merra2_ee": round(ee_2001["DP"], 2), "DP_power": round(p_period["DP"], 2),
    "DP_diff_pct": round(100 * (p_period["DP"] - ee_2001["DP"]) / ee_2001["DP"], 3),
    "RDD_merra2_ee": round(ee_2001["RDD_deg"], 2), "RDD_power": round(p_period["RDD_deg"], 2),
    "RDD_diff_deg": round(float(wind.angle_diff(p_period["RDD_deg"], ee_2001["RDD_deg"])), 3),
    "max_yearly_DP_diff_pct": round(float(cmp.DP_diff_pct.abs().max()), 3),
    "max_yearly_RDD_diff_deg": round(float(cmp.RDD_diff_deg.max()), 3),
}
print("\nwhole-period agreement:", POWER_AGREEMENT)
cmp.round(3).to_csv(OUT / "wind_power_crosscheck.csv", index=False)

In [ ]:
# WHAT THIS DOES: sensitivity check for the grid-cell choice: repeats DP/RDD for 2020-2024 using the pixel Earth
# Engine would sample at the centroid's coordinates (a different native cell) to show how much the cell matters.
naive_xy = merra2.ee_pixel_centre(T, *naive_ij)
naive_cache = RAW / f"merra2_u10v10_hourly_i{naive_ij[0]}_j{naive_ij[1]}_2020-2024.csv"
if naive_cache.exists() and not REFRESH:
    nh = pd.read_csv(naive_cache, parse_dates=["time"])
else:
    with ThreadPoolExecutor(5) as ex:
        nh = pd.concat(list(ex.map(lambda y: merra_year(y, naive_xy), range(2020, 2025))), ignore_index=True)
    nh.to_csv(naive_cache, index=False)
nh["time"] = pd.to_datetime(nh["time"], utc=True)
nr, nper = wind.fryberger_by_year(nh.time.dt.year.values, nh.U10M.values, nh.V10M.values)
sub = hourly[ok & (hourly.index.year >= 2020)]
cr, cper = wind.fryberger_by_year(sub.index.year, sub.U10M.values, sub.V10M.values)
nat_naive = (-90 + 0.5 * naive_ij[1], -180 + 0.625 * naive_ij[0])
print(f"2020-2024 used cell (native centre {cell['lat_c']}, {cell['lon_c']}): DP {cper['DP']:.1f}, RDD {cper['RDD_deg']:.1f}, RDP/DP {cper['RDP_DP']:.3f}")
print(f"2020-2024 naive cell (native centre {nat_naive[0]}, {nat_naive[1]}): DP {nper['DP']:.1f}, RDD {nper['RDD_deg']:.1f}, RDP/DP {nper['RDP_DP']:.3f}")

## What the wind numbers can and cannot support

**Result (MERRA-2, native cell centred 24.0 N 55.625 E, 2000-2024, no missing hours):** DP ≈ 66 VU/yr, which is **low** in Fryberger's classes. RDP ≈ 25 VU/yr, **RDD ≈ 100°** (sand moves toward the east, slightly south of east), and **RDP/DP ≈ 0.38**. Yearly DP ranges 44-88 VU and yearly RDD 46-174°. The exact values are printed above and written to `wind_summary.json`.

**Is the direction physically plausible?** The shamal alone would push sand toward the SE (~135°). RDD is about 35° north of that, so I broke the record down before reporting it:
- The sand rose is **bimodal**. About 54 % of DP comes from the WNW-NW sectors (292.5° and 315°): the shamal plus the afternoon north-westerly that peaks 08-13 UTC (12-17 local) and drives sand toward 95-135°. About 30 % comes from the SSE-SSW sectors (157.5-202.5°), mostly in the morning (03-05 UTC, 07-09 local), and drives sand toward the north (335-005°).
- By month, May-June is the most unimodal (RDD 124-129°, RDP/DP ≈ 0.8, the summer shamal). February-March drifts toward the NE (RDD 42-56°), and August-November contributes little.
- The vector sum of a NW mode and a S/SSE mode points roughly east, which is what we get. It is not a processing error: the direction maths passes the hand-checked tests, and NASA POWER gives the same numbers. RDP/DP ≈ 0.38 falls in Fryberger's intermediate band (0.3-0.8), as expected for a bimodal regime. The morning southerlies may be nocturnal drainage or land-breeze flow off the uplands to the east. That is an inference I have not verified.

**How far to trust it:**
- **Absolute DP is likely biased low.** MERRA-2 is a ~0.5° × 0.625° (about 55 × 64 km) reanalysis of hourly *means*, which damps gusts and short wind peaks. Because DP grows roughly with V³ above 12 kt, losing peaks lowers it strongly. Only about 9 % of hours exceed 12 kt here. Station anemometers would very likely give a higher DP, so the **"low" class is approximate**. Treat DP as a relative and interannual index, not a calibrated sand flux.
- **RDD and RDP/DP are the more robust outputs**, but not exact either. One year's RDD swings between 46° and 174°, so only the multi-year value is meaningful. Using the neighbouring cell to the north (the one Earth Engine returns at the centroid's own coordinates) shifts 2020-2024 RDD by about 13° (91.5° vs 104.2°) and DP by about 8 %. A fair uncertainty for RDD from the grid alone is about ±10-15°.
- The cell spans 23.75-24.25 N, 55.31-55.94 E. It reaches east to the longitude of Al Ain city (55.75 E), so it blends desert with the city and its surroundings. The box sits in its western half.
- **Grid georeferencing:** Earth Engine's MERRA-2 asset is offset from the native grid (0.25° in latitude, 0.1875° in longitude). Sampling at the centroid's own coordinates would silently read the cell to the north. The cell used here was checked hour-for-hour against NASA POWER.
- **NASA POWER agreement** (2001-2024, same MERRA-2 source, so this checks processing only): DP agrees to about 0.001 % and RDD to about 0.004° for the whole period, with yearly differences at most about 0.02 % and 0.06°. Of 210,384 common hours, one differs (2004-09-05 17 UTC, V10M +1.51 vs −4.47 m/s). It is below threshold either way, so DP is unaffected.

**For Phase 3:** compare the dune headings with RDD (~100°) *and* with the NW-mode downwind direction (~130°). In a bimodal regime the heading tells us which mode moves the sand, and a 30° disagreement would not by itself mean the tracking is wrong.

In [ ]:
# WHAT THIS DOES: writes public/data/<site>/wind_summary.json in the handoff §4 format with the real values (plus a
# note on conventions and data handling), then rebuilds public/data/sites.json so this file shows as "real".
out = {
    "location": {"lat": round(LAT_C, 4), "lon": round(LON_C, 4)},
    "period": "2000-2024",
    "source": "NASA MERRA-2 U10M/V10M hourly",
    "threshold_kt": 12,
    "DP": round(period["DP"], 1),
    "RDP": round(period["RDP"], 1),
    "RDD_deg": round(period["RDD_deg"], 1),
    "RDP_DP": round(period["RDP_DP"], 3),
    "dp_class": period["dp_class"],
    "yearly": [{"year": int(r.year), "DP": round(r.DP, 1), "RDP": round(r.RDP, 1), "RDD_deg": round(r.RDD_deg, 1)} for r in yearly.itertuples()],
    "sand_rose": [{"sector_deg": float(r.sector_from_deg), "dp": round(r.dp, 2)} for r in rose.itertuples()],
    "placeholder": False,
    "note": (
        "Fryberger & Dean (1979) drift potential, DP = sum V^2 (V - 12 kt)/100 * t with V the hourly-mean 10 m wind "
        "in knots and t each hour's share of its year in percent; DP, RDP are mean annual values in vector units (VU/yr). "
        "RDD_deg is the direction sand moves TO (resultant of downwind drift vectors), degrees clockwise from north. "
        "sand_rose.sector_deg is the direction the wind blows FROM (16 sectors of 22.5 deg centred on sector_deg); "
        "sand_rose.dp sums to DP. "
        f"Data: MERRA-2 tavg1_2d_slv_Nx via Earth Engine NASA/GSFC/MERRA/slv/2, native cell centred {cell['lat_c']} N "
        f"{cell['lon_c']} E (0.5 x 0.625 deg, about 55 x 63 km) containing the box centroid; Earth Engine's grid is "
        "offset from the native one, so the pixel was chosen by native index and checked against NASA POWER. "
        f"Missing hours: {int(miss.sum())} of {len(full)}. "
        f"NASA POWER cross-check 2001-2024: DP differs by {POWER_AGREEMENT['DP_diff_pct']} %, RDD by {POWER_AGREEMENT['RDD_diff_deg']} deg "
        "(same underlying MERRA-2, so this checks processing only). "
        "Caveat: a ~50 km reanalysis of hourly means damps gusts, so absolute DP is likely biased low against station "
        "anemometers and dp_class is approximate; RDD and RDP_DP are the more robust outputs. "
        "Computed by pipeline/01_scenes_and_wind.ipynb."
    ),
}
(PUB / "wind_summary.json").write_text(json.dumps(out, indent=1, allow_nan=False) + "\n")
idx = site_index.write_index()
me = next(s for s in idx["sites"] if s["id"] == SID)
print(f"wrote {(PUB / 'wind_summary.json').relative_to(config.ROOT)}")
print(f"public/data/sites.json: {SID} status={me['status']}; files: {me['files']}")
print(json.dumps({k: v for k, v in out.items() if k not in ('yearly', 'sand_rose', 'note')}, indent=1))